# Topics Comparison

Notebook for comparing `S3`, `FASTopic`, and a `TF-IDF` baseline on canonical Polymarket markets. The default text view is `full_description`, but every detector also supports `question` and `question_plus_full_description`.

In [1]:
from polymarket_research.data.canonical import CanonicalDatasetBuilder
from polymarket_research.data.raw import RawExternalCovariates, RawPolymarketHandle
from polymarket_research.utils import setup_root

from polymarket_research.research.topics_detector import (
    FASTopicDetector,
    S3TopicsDetector,
    TFIDFTopicBaseline,
    build_topic_input_frame,
    compare_topic_models,
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', None)
%matplotlib inline


/Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/_snmf.py:19: UserWarning: JAX not found, continuing with NumPy implementation.
  warnings.warn("JAX not found, continuing with NumPy implementation.")


In [2]:
REPO_ROOT = setup_root()
CANONICAL_CACHE_DIR = REPO_ROOT / "research_notebooks" / "running_artefacts_new" / "canonical_dataset"
MARKET_LIMIT = None
MARKET_ORDER = None  # one of: None, "latest", "largest"

if (CANONICAL_CACHE_DIR / "markets.parquet").exists() and MARKET_LIMIT is None and MARKET_ORDER is None:
    from polymarket_research.data.canonical import CanonicalDataset
    canonical = CanonicalDataset.from_parquet(CANONICAL_CACHE_DIR)
    print("Loaded canonical from parquet cache:", CANONICAL_CACHE_DIR)
else:
    raw_handle = RawPolymarketHandle()
    raw_bundle = raw_handle.load_bundle(
        include_market_universe=False,
        include_download_manifest=False,
        include_probabilities=True,
        include_raw_trades=False,
        market_limit=MARKET_LIMIT,
        market_order=MARKET_ORDER,
    )
    raw_external = RawExternalCovariates().load()
    canonical = CanonicalDatasetBuilder(
        raw_dataset=raw_bundle,
        raw_external=raw_external,
        resolved_only=True,
    ).build()
    if MARKET_LIMIT is None and MARKET_ORDER is None:
        canonical.save(CANONICAL_CACHE_DIR)
    print("Built canonical from SQLite")

all_markets = canonical.markets.copy()
probabilities = canonical.probabilities.copy()
print(f"Canonical resolved markets: {len(all_markets)}")
print("Market limit:", MARKET_LIMIT)
print("Market order:", MARKET_ORDER)
display(all_markets[["market_id", "question", "domain", "family_id", "volume_num"]].head(5))


Loaded canonical from parquet cache: /Users/sneddy/research/polymarket_research/research_notebooks/running_artefacts_new/canonical_dataset
Canonical resolved markets: 15662
Market limit: None
Market order: None


,market_id,question,domain,family_id,volume_num
0,1914442,"Will Hegseth say ""AI"" or ""Cyber"" during press conference?",unassigned,unassigned::::hegseth say ai or cyber during,38222.744674
1,1912772,"Trump announces US x Iran ceasefire end by April 8, 2026?",unassigned,unassigned::::trump announces us x iran ceasefire,38220.348645
2,1908626,Trump announces Hormuz deadline extension today?,unassigned,unassigned::::trump announces hormuz deadline extension today,987122.280619
3,1891746,"Will Trump say ""Strait"" or ""Hormuz"" during events with Rutte?",unassigned,unassigned::::trump say strait or hormuz during,27053.816874
4,1882888,Will Trump's remarks not air?,unassigned,unassigned::::trump s remarks not air,33098.189315


In [3]:
topic_markets = build_topic_input_frame(all_markets)
topic_markets["question_len"] = topic_markets["question"].fillna("").str.len()
topic_markets["full_description_len"] = topic_markets["full_description"].fillna("").str.len()

coverage = pd.DataFrame({
    "n_rows": [len(topic_markets)],
    "nonempty_question": [int(topic_markets["question"].fillna("").str.strip().ne("").sum())],
    "nonempty_full_description": [int(topic_markets["full_description"].fillna("").str.strip().ne("").sum())],
    "mean_question_len": [float(topic_markets["question_len"].mean())],
    "mean_full_description_len": [float(topic_markets["full_description_len"].mean())],
})
display(coverage)
display(topic_markets[["market_id", "question", "full_description", "volume_num"]].sample(2))

,n_rows,nonempty_question,nonempty_full_description,mean_question_len,mean_full_description_len
0,15662,15662,15662,57.048206,925.017558


,market_id,question,full_description,volume_num
7796,591854,Will Israel strike Syria by September 30?,"This market will resolve to ""Yes"" if Israel initiates a drone, missile, or air strike on Syrian soil or any Syrian embassy or consulate between September 9, 3 PM ET, and September 30, 2025, 11:59 PM ET. Otherwise, this market will resolve to ""No"". For the purposes of this market, a qualifying ""strike"" is defined as the use of aerial bombs, drones or missiles (including cruise or ballistic missiles) launched by Israeli military forces that impact Syrian ground territory or any official Syrian embassy or consulate (e.g., if a weapons depot on Syrian soil is hit by an Israeli missile, this market will resolve to ""Yes"") that is officially acknowledged by the Israeli government or a consensus of credible reporting. Missiles or drones which are intercepted and surface-to-air missile strikes will not be sufficient for a ""Yes"" resolution regardless of whether they land on Syrian territory or cause damage. Actions such as artillery fire, small arms fire, FPV or ATGM strikes directly, ground incursions, naval shelling, cyberattacks, or other operations conducted by Israeli ground operatives will not qualify. The resolution source will be a consensus of credible reporting. Will Israel strike Syria by...?",76279.059893
9399,561789,OpenAI browser in July?,"This market will resolve to ""Yes"" if OpenAI publicly releases a standalone web browser that is intended for general web browsing and is available for use by the public in at least one country or region by July 31, 2025, 11:59 PM ET. Otherwise, this market will resolve to ""No"". For this market to resolve to ""Yes,"" such a browser must be launched and publicly accessible, including via open beta or open rolling free waitlist signups. A closed beta or any form of private access will not suffice. The release must be clearly defined and publicly announced by OpenAI as being accessible to the general public. Browser extensions, plugins, or integrations within third-party browsers do not count; it must be a standalone browser application. The primary resolution source will be official announcements from OpenAI. If there is ambiguity, a consensus of credible reporting will be used. OpenAI browser in July?",100165.580510


## Run Config

Use `TEXT_MODE = "full_description"` for the description-centric view. Switch to `"question"` or `"question_plus_full_description"` to compare text views.

In [4]:
TEXT_MODE = "full_description"
N_TOPICS = 10
MIN_DF = 5
MAX_DF = 0.5
TOP_TERMS = 8
FASTOPIC_EPOCHS = 60
SAMPLE_SIZE = None
SORT_BY = "volume_num"
REDUCERS = ["umap", "tsne"]

work_markets = topic_markets.copy()
work_markets = work_markets.loc[work_markets[TEXT_MODE].fillna("").astype(str).str.strip().ne("")].copy()
if SORT_BY in work_markets.columns:
    work_markets = work_markets.sort_values(SORT_BY, ascending=False, kind="stable")
if SAMPLE_SIZE is not None:
    work_markets = work_markets.head(int(SAMPLE_SIZE)).copy()

print("Rows used for modeling:", len(work_markets))
display(work_markets[["market_id", "question", "full_description", "volume_num"]].sample(2))

Rows used for modeling: 15662


,market_id,question,full_description,volume_num
14746,520456,Will Saquon Barkley score the first touchdown of Super Bowl LIX?,"This market will resolve according to the player/unit to score the first touchdown of Super Bowl LIX. Any rushing or receiving touchdowns will count. Passing touchdowns will not qualify. Special teams and defensive touchdowns will count for the listed D/ST unit, not the individual player who scores them. If this game is delayed or postponed past February 28, 2025, 11:59 PM ET, this market will resolve to “No TD”. The resolution source for this market will be the official broadcast Super Bowl LIX on Fox, however a consensus of credible repotting may also be used. First Touchdown Scorer?",38860.297387
12983,530430,Will Oscar Piastri win the 2025 Japanese Grand Prix?,"This market will resolve according to the winner of the 2025 Japan Grand Prix scheduled for April 6, 2025. If the 2025 Japanese Grand Prix is canceled or rescheduled to a date after April 28, 2025, this market will resolve to “Other.” The resolution source will be the official Formula 1 website and credible sports news reporting. Japan Grand Prix Winner",50985.489452


In [ ]:
detectors = [
    S3TopicsDetector(
        n_topics=N_TOPICS,
        text_mode=TEXT_MODE,
        min_df=MIN_DF,
        max_df=MAX_DF,
        top_terms=TOP_TERMS,
    ),
    FASTopicDetector(
        n_topics=N_TOPICS,
        text_mode=TEXT_MODE,
        min_df=MIN_DF,
        max_df=MAX_DF,
        top_terms=TOP_TERMS,
        n_epochs=FASTOPIC_EPOCHS,
    ),
    TFIDFTopicBaseline(
        n_topics=N_TOPICS,
        text_mode=TEXT_MODE,
        min_df=MIN_DF,
        max_df=MAX_DF,
        top_terms=TOP_TERMS,
    ),
]

results, comparison_summary = compare_topic_models(work_markets, detectors)
display(comparison_summary)

Output()

[18:51:29] Documents encoded.                                                                         ]8;id=869316;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py\decomp.py]8;;\:]8;id=307863;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py#144\144]8;;\

/Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/sklearn/decomposition/_fastica.py:132: 
ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(

           Decomposition done.                                                                        ]8;id=268767;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py\decomp.py]8;;\:]8;id=785102;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py#151\151]8;;\

[18:51:30] Term extraction done.                                                                      ]8;id=524684;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py\decomp.py]8;;\:]8;id=336750;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py#154\154]8;;\

[18:51:33] Vocabulary encoded.                                                                        ]8;id=686252;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py\decomp.py]8;;\:]8;id=732295;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py#164\164]8;;\

           Model fitting done.                                                                        ]8;id=225833;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py\decomp.py]8;;\:]8;id=793546;file:///Users/sneddy/anaconda3/envs/polymarket/lib/python3.14/site-packages/turftopic/models/decomp.py#184\184]8;;\

Output()

In [ ]:
for model_name, result in results.items():
    print("=" * 100)
    print(model_name)
    display(result.topic_summary()[["topic_id", "topic_label", "topic_size"]])

In [ ]:
for model_name, result in results.items():
    fig, axes, projected = result.plot_2d_with_topic_map(
        reducer="umap",
        random_state=0,
        legend_mode="full",
        show_table=False,
        figsize=(18, 10),
    )
    plt.show()

In [ ]:
for model_name, result in results.items():
    fig, axes, projected = result.plot_2d_with_topic_map(
        reducer="tsne",
        random_state=0,
        table_position="bottom",
        figsize=(15, 10),
    )
    plt.show()

In [ ]:
TOP_DOCS_PER_TOPIC = 5
for model_name, result in results.items():
    print("=" * 100)
    print(f"Representative documents for {model_name}")
    display(
        result.representative_documents(top_n=TOP_DOCS_PER_TOPIC)[
            ["market_id", "topic_id", "topic_confidence", "topic_label", "question", "full_description"]
        ]
    )

In [ ]:
# Optional: compare question vs description-centric text on S3 only.
question_results, question_summary = compare_topic_models(
    work_markets,
    [
        S3TopicsDetector(n_topics=N_TOPICS, text_mode="question", min_df=MIN_DF, max_df=MAX_DF, top_terms=TOP_TERMS),
        S3TopicsDetector(n_topics=N_TOPICS, text_mode="full_description", min_df=MIN_DF, max_df=MAX_DF, top_terms=TOP_TERMS),
        S3TopicsDetector(n_topics=N_TOPICS, text_mode="question_plus_full_description", min_df=MIN_DF, max_df=MAX_DF, top_terms=TOP_TERMS),
    ],
)
display(question_summary)
for model_name, result in question_results.items():
    print(model_name, result.text_mode)
    display(result.topic_summary()[["topic_id", "topic_label", "topic_size"]].head(10))